<a href="https://colab.research.google.com/github/Rasya-ai-web/Machine-Learning-Lab/blob/main/ML_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

STUDENT PERFORMANCE DATA PREPROCESSING


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

DATA LOADING


In [ ]:
df = pd.read_csv("student_performance_updated_1000.csv")

print("Dataset loaded successfully!")
print("Original Shape:", df.shape)

Dataset loaded successfully!
Original Shape: (1000, 12)


 DATA UNDERSTANDING

In [ ]:
print("\n--- First 5 Rows ---")
print(df.head())

print("\n--- Dataset Information ---")
df.info()

print("\n--- Statistical Summary ---")
print(df.describe(include="all"))

print("\n--- Missing Values ---")
print(df.isnull().sum())

print("\n--- Duplicate Rows ---")
print(df.duplicated().sum())


--- First 5 Rows ---
   StudentID     Name  Gender  AttendanceRate  StudyHoursPerWeek  \
0        1.0     John    Male            85.0               15.0   
1        2.0    Sarah  Female            90.0               20.0   
2        3.0     Alex    Male            78.0               10.0   
3        4.0  Michael    Male            92.0               25.0   
4        5.0     Emma  Female             NaN               18.0   

   PreviousGrade  ExtracurricularActivities ParentalSupport  FinalGrade  \
0           78.0                        1.0            High        80.0   
1           85.0                        2.0          Medium        87.0   
2           65.0                        0.0             Low        68.0   
3           90.0                        3.0            High        92.0   
4           82.0                        2.0          Medium        85.0   

   Study Hours  Attendance (%) Online Classes Taken  
0          4.8            59.0                False  
1         

COLUMN STANDARDIZATION

In [ ]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("%", "percent", regex=False)
)

print("\n--- Standardized Columns ---")
print(df.columns.tolist())


--- Standardized Columns ---
['studentid', 'name', 'gender', 'attendancerate', 'studyhoursperweek', 'previousgrade', 'extracurricularactivities', 'parentalsupport', 'finalgrade', 'study_hours', 'attendance_(percent)', 'online_classes_taken']


 REMOVING UNNECESSARY COLUMNS

In [ ]:
unnecessary_columns = ["studentid", "name"]

df.drop(
    columns=unnecessary_columns,
    errors="ignore",
    inplace=True
)

print("\n--- After Removing Unnecessary Columns ---")
print(df.columns.tolist())


--- After Removing Unnecessary Columns ---
['gender', 'attendancerate', 'studyhoursperweek', 'previousgrade', 'extracurricularactivities', 'parentalsupport', 'finalgrade', 'study_hours', 'attendance_(percent)', 'online_classes_taken']


HANDLING INVALID VALUES

In [ ]:

df.replace([np.inf, -np.inf], np.nan, inplace=True)

for column in ["attendancerate", "attendance_percent"]:
    if column in df.columns:
        df.loc[
            (df[column] < 0) | (df[column] > 100),
            column
        ] = np.nan

for column in ["previousgrade", "finalgrade"]:
    if column in df.columns:
        df.loc[
            (df[column] < 0) | (df[column] > 100),
            column
        ] = np.nan


for column in ["studyhoursperweek", "study_hours"]:
    if column in df.columns:
        df.loc[
            df[column] < 0,
            column
        ] = np.nan

print("\nInvalid values handled.")


Invalid values handled.


REMOVING DUPLICATES

In [ ]:
before = len(df)

df.drop_duplicates(inplace=True)

after = len(df)

print("\nDuplicates removed:", before - after)
print("Shape after removing duplicates:", df.shape)


Duplicates removed: 0
Shape after removing duplicates: (1000, 10)


ENCODING

In [ ]:


for column in df.select_dtypes(include=["bool"]).columns:
    df[column] = df[column].astype(int)

categorical_columns = df.select_dtypes(
    include=["object"]
).columns

df = pd.get_dummies(
    df,
    columns=categorical_columns,
    drop_first=True,
    dtype=int
)

print("\nEncoding completed.")
print("Shape after encoding:", df.shape)


Encoding completed.
Shape after encoding: (1000, 11)


TRAIN-TEST SPLIT

In [ ]:
target_column = "finalgrade"

if target_column not in df.columns:
    raise ValueError(
        "Target column 'finalgrade' was not found. "
        "Check the column name in your dataset."
    )

X = df.drop(columns=[target_column])
y = df[target_column]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("\n--- Train-Test Split ---")
print("Training data:", X_train.shape)
print("Testing data :", X_test.shape)


--- Train-Test Split ---
Training data: (800, 10)
Testing data : (200, 10)


 MISSING VALUE IMPUTATION
    AFTER TRAIN-TEST SPLIT

In [ ]:
numeric_columns = X_train.select_dtypes(
    include=["int64", "float64"]
).columns

numeric_imputer = SimpleImputer(strategy="median")

X_train[numeric_columns] = numeric_imputer.fit_transform(
    X_train[numeric_columns]
)

X_test[numeric_columns] = numeric_imputer.transform(
    X_test[numeric_columns]
)

# Categorical/encoded features
other_columns = X_train.columns.difference(
    numeric_columns
)

if len(other_columns) > 0:

    categorical_imputer = SimpleImputer(
        strategy="most_frequent"
    )

    X_train[other_columns] = categorical_imputer.fit_transform(
        X_train[other_columns]
    )

    X_test[other_columns] = categorical_imputer.transform(
        X_test[other_columns]
    )

print("\nMissing value imputation completed.")

print("Missing values in training data:")
print(X_train.isnull().sum().sum())

print("Missing values in testing data:")
print(X_test.isnull().sum().sum())



Missing value imputation completed.
Missing values in training data:
0
Missing values in testing data:
0


FEATURE SCALING   FIT ONLY ON TRAINING DATA

In [ ]:

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)


X_train_scaled = pd.DataFrame(
    X_train_scaled,
    columns=X_train.columns,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    X_test_scaled,
    columns=X_test.columns,
    index=X_test.index
)

print("\nFeature scaling completed.")


Feature scaling completed.


SAVING THE CLEANED DATASET

In [ ]:


train_cleaned = pd.concat(
    [X_train_scaled, y_train],
    axis=1
)


test_cleaned = pd.concat(
    [X_test_scaled, y_test],
    axis=1
)

train_cleaned.to_csv(
    "student_performance_cleaned_train.csv",
    index=False
)

test_cleaned.to_csv(
    "student_performance_cleaned_test.csv",
    index=False
)

FINAL CHECK

In [ ]:

print("\n============================================")
print("PREPROCESSING COMPLETED SUCCESSFULLY")
print("============================================")

print("\nFinal Training Shape:",
      train_cleaned.shape)

print("Final Testing Shape:",
      test_cleaned.shape)

print("\nMissing values in training:",
      train_cleaned.isnull().sum().sum())

print("Missing values in testing:",
      test_cleaned.isnull().sum().sum())

print("\nSaved files:")
print("student_performance_cleaned_train.csv")
print("student_performance_cleaned_test.csv")


PREPROCESSING COMPLETED SUCCESSFULLY

Final Training Shape: (800, 11)
Final Testing Shape: (200, 11)

Missing values in training: 28
Missing values in testing: 12

Saved files:
student_performance_cleaned_train.csv
student_performance_cleaned_test.csv
